# **Monte Carlo Tree Search (MCTS)**

### Summer School practical session — planning on FrozenLake

This notebook is meant to be **self-contained**: it introduces the main ideas of MCTS and then asks you to implement the core loop.

We will use a small grid-world, **FrozenLake**, where the agent must reach the goal while avoiding holes.

By the end of the tutorial you should understand and implement:

1. how to sample from a simulator / transition model;
2. how to represent a search tree;
3. how **selection** uses optimism, similarly to UCB in bandits;
4. how **expansion** grows the tree one node at a time;
5. how **rollouts** estimate the value of unexplored states;
6. how **backpropagation** updates the statistics of the visited nodes;
7. how to choose the action to execute at the root.

The style is intentionally close to the stochastic bandit notebooks: explicit classes, simple loops, and small exercises with `...` to complete.

In [2]:
import math
from dataclasses import dataclass, field

import numpy as np
import matplotlib.pyplot as plt

# If you run this notebook on Colab and Gymnasium is missing, uncomment:
# %pip -q install gymnasium[toy-text]

# Install moviepy to visualize the bahavior of the final policy
# %pip install -q moviepy

try:
    import gymnasium as gym
    GYMNASIUM_AVAILABLE = True
except ImportError:
    gym = None
    GYMNASIUM_AVAILABLE = False

print("Gymnasium available:", GYMNASIUM_AVAILABLE)

Gymnasium available: True


## **0. What problem does MCTS solve?**

In many sequential decision problems, the agent cannot simply enumerate all future possibilities.

Suppose we are in a state $s_0$ and we can simulate the effect of actions. A full depth-$H$ planning tree has roughly

$$
1 + A + A^2 + \cdots + A^H
$$

nodes, where $A$ is the number of actions. This becomes large very quickly.

**Monte Carlo Tree Search** (MCTS) avoids building the full tree. Instead, it builds an **asymmetric partial tree**: it spends more simulations on actions and states that look promising.

Each MCTS simulation has four phases:

```text
1. Selection:     follow the current tree using an optimistic score
2. Expansion:     add one new child to the tree
3. Rollout:       simulate randomly outside the tree
4. Backpropagate: update all nodes seen in the simulation
```

At the end, we execute the most visited action from the root.

### Why is there a bandit idea inside MCTS?

At each tree node, choosing which child to explore is a small exploration-exploitation problem:

- exploit children with high empirical value;
- explore children with few visits.

To cope with this tradeoff, the standard tree policy employs optimism, thus using an upper-confidence score:

$$
\mathrm{UCB}(s,a) = \widehat Q(s,a) + c \sqrt{\frac{\log N(s)}{N(s,a)}}.
$$

Here, $N(s)$ is the number of visits to the parent node, and $N(s,a)$ is the number of visits to the child reached by action $a$. The more the policy visits the parent node, the larger the confidence interval will get; however, if the policy keeps visiting the same child, the increase in the denominator in the square root will dominate. As a result, the confidence interval will shrink. The quantity $c$ is a fixed constant.

## **1. FrozenLake as a planning problem**

FrozenLake is a small grid-world.

- `S`: starting state;
- `F`: frozen safe cell;
- `H`: hole, terminal bad state;
- `G`: goal, terminal good state.

Actions are encoded as:

- `0`: left;
- `1`: down;
- `2`: right;
- `3`: up.

In this tutorial MCTS uses a **generative model** of the environment: given a state and an action, we can sample one possible next state.

For FrozenLake, the transition table has the form

```python
P[state][action] = [(probability, next_state, reward, done), ...]
```

The following cell uses Gymnasium if available. Otherwise it defines a tiny FrozenLake clone with the same transition-table interface.

In [3]:
DEFAULT_MAP_NAME = "4x4"

MAPS = {
    "4x4": [
        "SFFF",
        "FHFH",
        "FFFH",
        "HFFG",
    ],
    "8x8": [
        "SFFFFFFF",
        "FFFFFFFF",
        "FFFHFFFF",
        "FFFFFHFF",
        "FFFHFFFF",
        "FHHFFFHF",
        "FHFFHFHF",
        "FFFHFFFG",
    ],
}

# Same convention used by Gymnasium FrozenLake.
LEFT, DOWN, RIGHT, UP = 0, 1, 2, 3
ACTION_NAMES = {
    LEFT: "LEFT",
    DOWN: "DOWN",
    RIGHT: "RIGHT",
    UP: "UP",
}


class DiscreteActionSpace:
    def __init__(self, n, seed=0):
        self.n = n
        self.rng = np.random.default_rng(seed)

    def sample(self):
        return int(self.rng.integers(self.n))


class SimpleFrozenLakeEnv:
    """Small FrozenLake clone used when Gymnasium is not installed."""

    def __init__(self, map_name="4x4", is_slippery=False, seed=0):
        self.map_name = map_name
        self.is_slippery = is_slippery
        self.desc = np.asarray(MAPS[map_name], dtype="c")
        self.nrow, self.ncol = self.desc.shape
        self.num_states = self.nrow * self.ncol
        self.action_space = DiscreteActionSpace(4, seed=seed)
        self.rng = np.random.default_rng(seed)
        self.unwrapped = self
        self.P = self._build_transition_model()
        self.state = 0

    def _to_state(self, row, col):
        return row * self.ncol + col

    def _inc(self, row, col, action):
        if action == LEFT:
            col = max(col - 1, 0)
        elif action == DOWN:
            row = min(row + 1, self.nrow - 1)
        elif action == RIGHT:
            col = min(col + 1, self.ncol - 1)
        elif action == UP:
            row = max(row - 1, 0)
        return row, col

    def _outcome(self, state, action):
        row, col = divmod(state, self.ncol)
        row, col = self._inc(row, col, action)
        next_state = self._to_state(row, col)
        tile = self.desc[row, col]
        done = tile in (b"H", b"G")
        reward = float(tile == b"G")
        return next_state, reward, done

    def _build_transition_model(self):
        P = {s: {a: [] for a in range(4)} for s in range(self.num_states)}

        for s in range(self.num_states):
            row, col = divmod(s, self.ncol)
            tile = self.desc[row, col]

            for a in range(4):
                if tile in (b"H", b"G"):
                    P[s][a] = [(1.0, s, 0.0, True)]
                    continue

                if self.is_slippery:
                    # As in FrozenLake: intended direction plus the two perpendicular directions.
                    candidate_actions = [(a - 1) % 4, a, (a + 1) % 4]
                    prob = 1.0 / 3.0
                else:
                    candidate_actions = [a]
                    prob = 1.0

                outcomes = []
                for actual_action in candidate_actions:
                    next_state, reward, done = self._outcome(s, actual_action)
                    outcomes.append((prob, next_state, reward, done))
                P[s][a] = outcomes

        return P

    def reset(self, seed=None):
        if seed is not None:
            self.rng = np.random.default_rng(seed)
            self.action_space = DiscreteActionSpace(4, seed=seed)
        self.state = 0
        return self.state, {}

    def step(self, action):
        outcomes = self.P[int(self.state)][int(action)]
        probs = [x[0] for x in outcomes]
        idx = int(self.rng.choice(len(outcomes), p=probs))
        _, next_state, reward, done = outcomes[idx]
        self.state = int(next_state)
        return int(next_state), float(reward), bool(done), False, {}


def make_env(map_name=DEFAULT_MAP_NAME, is_slippery=False, seed=0, render_mode=None):
    if GYMNASIUM_AVAILABLE:
        return gym.make("FrozenLake-v1", map_name=map_name, is_slippery=is_slippery, render_mode=render_mode)
    return SimpleFrozenLakeEnv(map_name=map_name, is_slippery=is_slippery, seed=seed)


def show_map(env):
    desc = np.asarray(env.unwrapped.desc, dtype="c")
    for row in desc:
        print(" ".join(x.decode("utf-8") for x in row))


def state_to_position(env, state):
    ncol = int(env.unwrapped.ncol)
    return divmod(int(state), ncol)


In [4]:
env = make_env(map_name=DEFAULT_MAP_NAME, is_slippery=False, seed=0)
show_map(env)

state, _ = env.reset(seed=0)
print("\nInitial state:", state)
print("Available actions:", ACTION_NAMES)
print("Transition from state 0 with action RIGHT:", env.unwrapped.P[0][RIGHT])

S F F F
F H F H
F F F H
H F F G

Initial state: 0
Available actions: {0: 'LEFT', 1: 'DOWN', 2: 'RIGHT', 3: 'UP'}
Transition from state 0 with action RIGHT: [(1.0, 1, 0, False)]


## **2. The search tree**

MCTS stores statistics in a tree.

Each node corresponds to a state. For each node we store:

- `visits`: how many simulations passed through this node;
- `value_sum`: cumulative simulated return observed from this node;
- `mean_value = value_sum / visits`;
- `children`: already expanded actions;
- `untried_actions`: actions that have not been expanded yet.

A useful interpretation is:

```text
node.children[a] = node reached after taking action a
```

In FrozenLake the reward is sparse: the reward is `1` only when we reach the goal.

In [6]:
@dataclass
class Node:
    state: int
    parent: "Node | None" = None
    action_from_parent: int | None = None
    reward_from_parent: float = 0.0
    untried_actions: list[int] = field(default_factory=list)
    children: dict[int, "Node"] = field(default_factory=dict)
    visits: int = 0
    value_sum: float = 0.0

    @property
    def mean_value(self):
        if self.visits == 0:
            return 0.0
        return self.value_sum / self.visits

## **3. Implementing MCTS**

The class below contains the complete algorithmic skeleton. Your task is to fill the gaps following the TODO schedule.

Before coding, keep the four phases in mind.

- **Selection**. Start at the root. While the current node is non-terminal and fully expanded, move to the child with the largest UCB score. (This is how the algorithm favors *exploitation*.)
- **Expansion**. When we reach a node with at least one untried action, sample one of those actions and add the resulting state as a new child. (This is how *exploration* is favored instead.)
- **Rollout**. From the new state, simulate a simple default policy. Here the default policy is uniformly random.
- **Backpropagation**. Update visit counts and value sums from the expanded node back to the root.

In [ ]:
class FrozenLakeMCTS:
    def __init__(
        self,
        env,
        iterations=300,
        exploration_c=1.4,
        rollout_depth=40,
        seed=0,
        gamma=0.97,
    ):
        self.env = env
        self.iterations = iterations
        self.exploration_c = exploration_c
        self.rollout_depth = rollout_depth
        self.rng = np.random.default_rng(seed)
        self.gamma = gamma

        self.num_actions = env.action_space.n
        self.transitions = env.unwrapped.P
        self.nrow = int(env.unwrapped.nrow)
        self.ncol = int(env.unwrapped.ncol)
        self.desc = np.asarray(env.unwrapped.desc, dtype="c")

    def _available_actions(self):
        return list(range(self.num_actions))

    def _is_terminal(self, state):
        row, col = divmod(int(state), self.ncol)
        return self.desc[row, col] in (b"H", b"G")

    def _sample_next_state(self, state, action):
        """TODO 1: sample one successor from self.transitions[state][action].

        Hint: outcomes are tuples of the form
            (probability, next_state, reward, done).
        Use self.rng.choice with the list of probabilities.
        """

        outcomes = self.transitions[state][action]
        probs = [outcome[0] for outcome in outcomes]
        idx = self.rng.choice(p=probs)
        _, next_state, reward, done = outcomes[idx]
        return int(next_state), float(reward), bool(done)

    def _ucb_score(self, parent, child):
        """TODO 2: compute the UCB1 tree-policy score.

        If child.visits == 0, return +infinity.
        Otherwise return:
            child.mean_value + c * sqrt(log(parent.visits) / child.visits)
        """
        if child.visits == 0:
            return np.inf
        # bonus = ...
        return  child.mean_value + self.exploration_c * math.sqrt(math.log(parent.visits) / child.visits)

    def _select(self, node):
        """TODO 3: selection phase.

        Descend while the node is non-terminal, fully expanded, and has children.
        At each step, move to the child with largest UCB score.
        """
        # is terminal?
        state = node.state
        row , col = divmod(state, self.env.ncol)
        tile = self.env.desc[row,col]
        terminal = tile in (b"H", b"G")
        # terminal = self._is_terminal(state)
       
        # fully expanded?
        fully_expanded = len(node.untried_actions) == 0

        # has children?
        has_children = len(node.childrens) > 0

        # while False:  # TODO: replace False with the correct condition
        while not terminal and fully_expanded and has_children:
            scores = {}
            for action , child in node.children.items():
                scores[action] = self._ucb_score(node,child)
            best_action = max(scores, key=scores.get)
            node = node.children[best_action]
        return node
    
#     @dataclass
# class Node:
#     state: int
#     parent: "Node | None" = None
#     action_from_parent: int | None = None
#     reward_from_parent: float = 0.0
#     untried_actions: list[int] = field(default_factory=list)
#     children: dict[int, "Node"] = field(default_factory=dict)
#     visits: int = 0
#     value_sum: float = 0.0

#     @property
#     def mean_value(self):
#         if self.visits == 0:
#             return 0.0
#         return self.value_sum / self.visits

    def _expand(self, node):
        """TODO 4: expansion phase.

        Remove one still-untried action, sample the next state, create a child,
        store it in node.children[action], and return (child, immediate_reward).
        """
        action = ...
        next_state, reward, done = ...

        child = Node(
            state=...,
            parent=...,
            action_from_parent=...,
            reward_from_parent=...,
            untried_actions=...,
        )
        node.children[action] = child
        return child, reward

    def _rollout(self, state):
        """TODO 5: rollout phase.

        Simulate a uniformly random policy for at most rollout_depth steps.
        Stop early at terminal states. Accumulate discounted rewards.
        """
        total_return = 0.0
        discount = 1.0
        current_state = int(state)

        for depth in range(self.rollout_depth):
            if False:  # TODO: replace False with the terminal-state condition
                break

            action = ...
            next_state, reward, done = ...
            total_return += ...

            if done:
                break

            current_state = ...
            discount *= ...

        return float(total_return)

    def _backpropagate(self, node, reward):
        """TODO 6: backpropagation phase.

        Starting from node, move through parent pointers up to the root.
        Each visited node gets one additional visit and adds reward to value_sum.
        """
        while node is not None:
            node.visits += ...
            node.value_sum += ...
            node = ...

    def plan(self, root_state):
        root = Node(state=int(root_state), untried_actions=self._available_actions())

        for _ in range(self.iterations):
            # TODO 7: one complete MCTS simulation.
            # 1. Select a node.
            # 2. If it is terminal, backpropagate its terminal reward_from_parent.
            # 3. Otherwise, expand if possible.
            # 4. Run a rollout and backpropagate immediate_reward + gamma * rollout_return.
            node = ...

            if False:  # TODO: replace False with a terminal-state condition
                total_return = ...
                self._backpropagate(...)
            else:
                if False:  # TODO: replace False with an expansion condition
                    child, immediate_reward = ...
                    rollout_return = ...
                    total_return = ...
                    self._backpropagate(...)
                else:
                    rollout_return = ...
                    self._backpropagate(...)

        # TODO 8: choose the root child with the largest visit count.
        # If the root has no children, return LEFT as a defensive default.
        if False:
            return LEFT

        best_child = ...
        return int(best_child.action_from_parent)

In [ ]:
A = [1,2,3,4]
b = A.pop()
print(A,b)



[1, 2, 3] 4


## **4. Small tests for the exercises**

Run these cells as you complete the TODOs.

They are not exhaustive, but they catch most implementation mistakes before running full episodes.

In [ ]:
# Test for TODO 1: sampling the transition model
np.random.seed(0)
env = make_env(map_name=DEFAULT_MAP_NAME, is_slippery=False, seed=0)
planner = FrozenLakeMCTS(env, iterations=10, seed=0)

next_state, reward, done = planner._sample_next_state(0, RIGHT)
assert next_state == 1
assert reward == 0.0
assert done is False
print("TODO 1 passed.")

In [ ]:
# Test for TODO 2: UCB score
parent = Node(state=0, visits=10)
visited_child = Node(state=1, parent=parent, visits=5, value_sum=2.5)
unvisited_child = Node(state=2, parent=parent, visits=0, value_sum=0.0)

score = planner._ucb_score(parent, visited_child)
assert np.isfinite(score)
assert planner._ucb_score(parent, unvisited_child) == float("inf")
print("TODO 2 passed.")

In [ ]:
# Test for TODO 3: selection
root = Node(state=0, untried_actions=[], visits=10)
child_a = Node(state=1, parent=root, action_from_parent=RIGHT, visits=5, value_sum=1.0)
child_b = Node(state=4, parent=root, action_from_parent=DOWN, visits=2, value_sum=1.5)
root.children = {RIGHT: child_a, DOWN: child_b}

selected = planner._select(root)
assert selected in [child_a, child_b]
assert selected is not root
print("TODO 3 passed.")

In [ ]:
# Test for TODO 4: expansion
root = Node(state=0, untried_actions=[RIGHT])
child, reward = planner._expand(root)

assert RIGHT in root.children
assert child.parent is root
assert child.action_from_parent == RIGHT
assert child.state == 1
assert reward == 0.0
print("TODO 4 passed.")

In [ ]:
# Test for TODO 5: rollout
value = planner._rollout(0)
assert isinstance(value, float)
assert 0.0 <= value <= 1.0
print("TODO 5 passed.")

In [ ]:
# Test for TODO 6: backpropagation
root = Node(state=0)
child = Node(state=1, parent=root)
planner._backpropagate(child, reward=0.7)

assert child.visits == 1
assert root.visits == 1
assert np.isclose(child.value_sum, 0.7)
assert np.isclose(root.value_sum, 0.7)
print("TODO 6 passed.")

In [ ]:
# Test for TODO 7-8: complete planning loop
planner = FrozenLakeMCTS(env, iterations=50, seed=0)
action = planner.plan(0)
assert action in [LEFT, DOWN, RIGHT, UP]
print("TODO 7-8 passed. Selected action:", ACTION_NAMES[action])

## **5. Running full episodes**

Now we use MCTS as an online planner.

At each real environment state, we:

1. rebuild a tree with that state as root;
2. run a fixed number of MCTS simulations;
3. execute the most visited root action;
4. move to the next real environment state.

This is computationally heavier than tabular dynamic programming, but it is much more flexible when the state space is large and only simulation is available.

In [ ]:
def run_episode(env, planner, seed=0, max_steps=100):
    state, _ = env.reset(seed=seed)
    total_reward = 0.0
    trajectory = [int(state)]
    actions = []

    done = False
    steps = 0

    while not done and steps < max_steps:
        action = planner.plan(int(state))
        next_state, reward, terminated, truncated, _ = env.step(action)

        actions.append(int(action))
        trajectory.append(int(next_state))
        total_reward += float(reward)

        state = int(next_state)
        done = bool(terminated or truncated)
        steps += 1

    return total_reward, trajectory, actions


def evaluate_mcts(map_name=DEFAULT_MAP_NAME, is_slippery=False, iterations=100, episodes=20, seed=0):
    env = make_env(map_name=map_name, is_slippery=is_slippery, seed=seed)
    rewards = []

    for ep in range(episodes):
        planner = FrozenLakeMCTS(env=env, iterations=iterations, seed=seed + 1000 * ep)
        reward, trajectory, actions = run_episode(env, planner, seed=seed + ep)
        rewards.append(reward)

    return np.asarray(rewards)


class RandomPlanner:
    def __init__(self, env, seed=0):
        self.env = env
        self.rng = np.random.default_rng(seed)

    def plan(self, state):
        return int(self.rng.integers(self.env.action_space.n))


def evaluate_random(map_name=DEFAULT_MAP_NAME, is_slippery=False, episodes=20, seed=0):
    env = make_env(map_name=map_name, is_slippery=is_slippery, seed=seed)
    planner = RandomPlanner(env, seed=seed)
    rewards = []

    for ep in range(episodes):
        reward, trajectory, actions = run_episode(env, planner, seed=seed + ep)
        rewards.append(reward)

    return np.asarray(rewards)

In [ ]:
env = make_env(map_name=DEFAULT_MAP_NAME, is_slippery=False, seed=0)
planner = FrozenLakeMCTS(env=env, iterations=100, seed=0)

reward, trajectory, actions = run_episode(env, planner, seed=0)

print("Reward:", reward)
print("Trajectory:", trajectory)
print("Actions:", [ACTION_NAMES[a] for a in actions])

## **6. Experiment: more simulations usually help**

The main computational budget of MCTS is the number of simulations run at every decision point.

Try to answer:

- What happens when `iterations` is too small?
- What changes when the lake is slippery?
- Why is the 8x8 map much harder than the 4x4 map?

In [ ]:
iteration_grid = [5, 25, 75]
episodes = 8
success_rates = []

for iterations in iteration_grid:
    rewards = evaluate_mcts(
        map_name=DEFAULT_MAP_NAME,
        is_slippery=False,
        iterations=iterations,
        episodes=episodes,
        seed=123,
    )
    success_rate = np.mean(rewards)
    success_rates.append(success_rate)
    print(f"iterations={iterations:3d} | success rate={success_rate:.2f}")

random_rewards = evaluate_random(map_name=DEFAULT_MAP_NAME, is_slippery=False, episodes=episodes, seed=123)
print(f"random policy | success rate={np.mean(random_rewards):.2f}")

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(iteration_grid, success_rates, marker="o", label="MCTS")
plt.axhline(np.mean(random_rewards), linestyle="--", label="Random")
plt.xlabel("MCTS iterations per decision")
plt.ylabel("Success rate")
plt.ylim(-0.05, 1.05)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## **9. Episode Visualization**

The code below allows us to visualize the trajectory performed by the MCTS algorithm.

In [5]:
# Install moviepy if not already installed
%pip install -q moviepy

In [ ]:
import base64
from IPython.display import HTML, display
from moviepy.video.io.ImageSequenceClip import ImageSequenceClip

def create_and_display_video(frames, filename='frozen_lake_episode.mp4', fps=4):
    """Creates an MP4 video from a list of frames and displays it in the notebook."""
    if not frames:
        print("No frames to create a video.")
        return

    # Ensure frames are uint8 type for moviepy
    frames_uint8 = [f.astype(np.uint8) for f in frames]

    clip = ImageSequenceClip(frames_uint8, fps=fps)
    clip.write_videofile(filename, codec='libx264', audio_codec='aac', logger=None)

    video_path = filename
    video_base64 = base64.b64encode(open(video_path, 'rb').read()).decode()
    display(HTML(f'''
    <video width="320" height="240" controls>
        <source src="data:video/mp4;base64,{video_base64}" type="video/mp4">
        Your browser does not support the video tag.
    </video>
    '''))
    print(f"Video saved as {filename}")

In [ ]:
def run_episode_and_render(env, planner, seed=0, max_steps=100):
    """Runs an episode and captures frames for visualization."""
    state, _ = env.reset(seed=seed)
    total_reward = 0.0
    trajectory = [int(state)]
    actions = []
    frames = []

    done = False
    steps = 0

    # Capture initial frame
    frames.append(env.render())

    while not done and steps < max_steps:
        action = planner.plan(int(state))
        next_state, reward, terminated, truncated, _ = env.step(action)

        actions.append(int(action))
        trajectory.append(int(next_state))
        total_reward += float(reward)

        state = int(next_state)
        done = bool(terminated or truncated)
        steps += 1

        # Capture frame after each step
        frames.append(env.render())

    return total_reward, trajectory, actions, frames

# Re-create environment with render_mode='rgb_array'
# Note: SimpleFrozenLakeEnv does not support render_mode, so this will only work with gym.make
env_render = make_env(map_name=DEFAULT_MAP_NAME, is_slippery=False, seed=0, render_mode='rgb_array')

# Re-initialize planner for consistency (or use existing one if suitable)
planner_render = FrozenLakeMCTS(env=env_render, iterations=100, seed=0)

# Run an episode with rendering
reward_render, trajectory_render, actions_render, frames_render = run_episode_and_render(env_render, planner_render, seed=0)

print("Reward (rendered episode):", reward_render)
print("Trajectory (rendered episode):", trajectory_render)
print("Actions (rendered episode):", [ACTION_NAMES[a] for a in actions_render])

# Create and display the video
create_and_display_video(frames_render, filename='frozen_lake_mcts_episode.mp4')

## **7. Stochastic transitions: slippery FrozenLake**

When `is_slippery=True`, the action you choose is not always the action that is executed.

This makes planning harder because the same action from the same state can lead to different next states. The MCTS implementation does not need to change: the simulator already samples from the correct transition probabilities.

In [ ]:
for slippery in [False, True]:
    rewards = evaluate_mcts(
        map_name=DEFAULT_MAP_NAME,
        is_slippery=slippery,
        iterations=100,
        episodes=10,
        seed=7,
    )
    print(f"slippery={slippery:<5} | success rate={np.mean(rewards):.2f}")

## **8. Optional extensions**

Students who finish early can try one of these:

1. **Heuristic rollout:** instead of uniformly random actions, bias rollouts toward the goal.
2. **Tree reuse:** after executing an action, keep the corresponding subtree instead of rebuilding from scratch.
3. **Different exploration constants:** compare `exploration_c = 0.1, 0.5, 1.4, 3.0`.
4. **8x8 map:** increase the map size and discuss what extra budget is needed.
5. **Discount factor:** compare `gamma = 0.9, 0.97, 1.0`.

In [ ]:
# Optional exercise: compare exploration constants.
# Fill the list and run the experiment.
exploration_grid = [...]

for c in exploration_grid:
    env = make_env(map_name=DEFAULT_MAP_NAME, is_slippery=False, seed=0)
    rewards = []
    for ep in range(8):
        planner = FrozenLakeMCTS(env=env, iterations=75, exploration_c=c, seed=ep)
        reward, _, _ = run_episode(env, planner, seed=ep)
        rewards.append(reward)
    print(f"c={c} | success rate={np.mean(rewards):.2f}")

## **Takeaways**

MCTS is useful when:

- we can simulate the environment;
- the full planning tree is too large to enumerate;
- we need an anytime method that improves with more simulations.

The four core ingredients are:

1. **Selection:** use optimism to move through the current tree;
2. **Expansion:** add one new child;
3. **Rollout:** estimate the value of the new state;
4. **Backpropagation:** update all ancestors with the simulated return.

The connection with bandits is local but important: every internal node faces an exploration-exploitation problem over its children.